## Creacion esquema y tablas

In [0]:
%sql
-- Usar tu catálogo
USE CATALOG bootcamp;
-- Crear esquema Landing
CREATE SCHEMA IF NOT EXISTS bootcamp.landing
COMMENT 'Esquema para archivos';
-- Crear esquema Raw
CREATE SCHEMA IF NOT EXISTS bootcamp.bronze
COMMENT 'Esquema para datos crudos sin procesar';
-- Verificar
SHOW SCHEMAS;

In [0]:
%sql
-- ============================================================
-- PASO 3: Crear the Volume para archivos
-- ============================================================
-- Crear volume tipo MANAGED (Databricks administra el storage)
CREATE VOLUME IF NOT EXISTS bootcamp.landing.archivos
COMMENT 'Volume para almacenar archivos CSV crudos';
-- Verificar que se creó
SHOW VOLUMES IN bootcamp.landing;

In [0]:
%sql
-- ============================================================
-- PASO 4: Verificar que el archivo se subió correctamente
-- ============================================================
-- Listar archivos en el volume
LIST '/Volumes/bootcamp/landing/archivos/';

In [0]:
%sql
-- Leer directo del archivo es posible, y es útil también para analizarlo
SELECT * FROM read_files(
'/Volumes/bootcamp/landing/archivos/properties_raw.csv',
format => 'csv',
header => true
)

In [0]:
%sql
-- ============================================================
-- PASO 5: Crear la tabla properties_bronze
-- ============================================================
-- Primero, eliminamos la tabla si existe (para poder recrearla)
DROP TABLE IF EXISTS bootcamp.bronze.properties_bronze;
-- Crear tabla EXTERNA leyendo el CSV y nos quedamos solo con los registros que tienen url válida
CREATE TABLE bootcamp.bronze.properties_bronze
SELECT * FROM read_files(
'/Volumes/bootcamp/landing/archivos/properties_raw.csv',
format => 'csv',
header => true
)
where url like 'https%' --Filtramos por aquellos links que sean consistentes
;

In [0]:
%sql
-- Verificar que la tabla se creó correctamente
SHOW TABLES IN bootcamp.bronze;

# EDA PARTE 1 - EXPLORACION INICIAL

In [0]:
%sql
-- E1.1
-- CUANTOS REGISTROS CONTIENE LA TABLA bootcamp.bronze.properties_bronze?
select count(*) from bootcamp.bronze.properties_bronze;

In [0]:
%sql
-- E1.2
-- Que columnas posee la tabla?

DESCRIBE bootcamp.bronze.properties_bronze;

In [0]:
%sql

--E1.3
-- Muestra de datos
select
    id,
    ubicacion,
    precio,
    expensas,
    tipo_de_operacion,
    moneda,
    ambientes,
    metros_cuadrados_totales,
    antiguedad,
    estado,
    zona
from bootcamp.bronze.properties_bronze
limit 10;

# EDA PARTE 2 - ANALISIS VALORES NULOS

In [0]:
%sql
-- E2.1
-- Contar nulos por columna

select
    count(*) as total_registros,
    total_registros - count(precio) as total_null_col_precio,
    total_registros - count(expensas) as total_null_col_expensas,
    total_registros - count(tipo_de_operacion) as total_null_tipo_operacion,
    total_registros - count(moneda) as total_null_moneda,
    total_registros - count(ambientes) as total_null_ambientes,
    total_registros - count(metros_cuadrados_totales) as total_null_m2_totales,
    total_registros - count(metros_cuadrados_cubiertos) as total_null_m2_cub,
    total_registros - count(orientacion_cardinal) as total_null_orientacion_cardinal,
    total_registros - count(piso) as total_null_piso,
    total_registros - count(cochera) as total_null_cochera,
    total_registros - count(antiguedad) as total_null_antiguedad,
    total_registros - count(estado) as total_null_estado,
    total_registros - count(zona) as total_null_zona
    from bootcamp.bronze.properties_bronze;
    

In [0]:
-- E2.2 y E2.3
-- Porcentaje de nulos

with porc_nulos as (
    select
    count(*) as cant_reg,
    (cant_reg - count(tr.precio)) *100.0/ cant_reg as porc_null_precio,
    (cant_reg - count(tr.expensas))*100.0/cant_reg as porc_null_expensas,
    (cant_reg - count(tr.ambientes))*100.0/cant_reg as porc_null_ambientes,
    (cant_reg - count(tr.metros_cuadrados_totales))*100.0/cant_reg as porc_null_m2_totales,
    (cant_reg - count(tr.metros_cuadrados_cubiertos))*100.0/cant_reg as porc_null_m2_cub,
    (cant_reg - count(tr.orientacion_cardinal))*100.0/cant_reg as porc_null_orientacion_cardinal, 
    (cant_reg - count(tr.antiguedad))*100.0/cant_reg as porc_null_antiguedad
    from bootcamp.bronze.properties_bronze as tr

)


select
    columna, porcentaje_null
from porc_nulos
UNPIVOT (
    porcentaje_null FOR columna IN (
        porc_null_precio, porc_null_expensas, porc_null_ambientes, 
        porc_null_m2_totales, porc_null_m2_cub,
        porc_null_orientacion_cardinal, porc_null_antiguedad
    )
)
where porcentaje_null>50.0;

# EDA PARTE 3 - CARDINALIDAD Y DISTRIBUCION

In [0]:
%sql
-- E3.1 - Distribución de tipo de operación - ¿Qué valores únicos hay en tipo_de_operacion? ¿Cuántos
-- registros hay de cada tipo? ¿Qué porcentaje representa cada uno? Pista: Usa GROUP BY, COUNT(*), y
-- SUM(COUNT(*)) OVER() para calcular porcentajes
WITH reg as (
    SELECT
        count(*) as total_registros
    from bootcamp.bronze.properties_bronze
)
select
    tipo_de_operacion,
    count(*) as total_registros,
    round(count(*) * 100.0 / sum(count(*)) over(), 4) as porcentaje
    
from bootcamp.bronze.properties_bronze

group by tipo_de_operacion
order by tipo_de_operacion;

In [0]:
%sql

--E3.2 - Distribución de moneda - ¿Qué monedas hay en el dataset? ¿Cuántos registros hay de cada una?
-- ¿Hay alguna moneda que no esperabas? Pista: GROUP BY moneda con conteo y porcentaje


select 
    moneda,
    count(*) as total_registros,
    round(count(*) * 100.0 / sum(count(*)) over(), 4) as porcentaje
from bootcamp.bronze.properties_bronze
group by moneda
order by moneda;

In [0]:
%sql
--E3.3 
-- Distribución de ambientes - ¿Cuántas propiedades hay por cantidad de ambientes? ¿Cuál es la
-- distribución? Ordena por cantidad de ambientes. Pista: GROUP BY ambientes ORDER BY ambientes

select 
    ambientes,
    count(*) as total_registros,
    round(count(*) * 100.0 / sum(count(*)) over(), 4) as porcentaje
from bootcamp.bronze.properties_bronze
group by ambientes
order by ambientes;


In [0]:
%sql
-- E3.4
-- Top zonas - ¿Cuáles son las 15 zonas con más propiedades? Muestra zona, cantidad y porcentaje
-- del total. Pista: GROUP BY zona, COUNT(*), porcentaje, ORDER BY cantidad DESC LIMIT 15

select
    zona,
    count(*) as total_propiedades,
    round(count(*) * 100.0 / sum(count(*)) over(), 4) as porcentaje
from bootcamp.bronze.properties_bronze
group by zona
order by total_propiedades desc;
-- limit 15;

In [0]:
%sql
-- E3.5
-- Distribución de estado - ¿Qué valores hay en la columna estado? ¿Cuántas propiedades hay de
-- cada estado? Pista: GROUP BY estado ORDER BY cantidad DESC

select
    estado,
    count(*) as total_propiedades
from bootcamp.bronze.properties_bronze
group by estado
order by estado;
